In [1]:
import pandas as pd # Used for loading, cleaning, manipulating, and analyzing structured tabular data (like CSV or Excel files) using DataFrames
from scipy.stats import false_discovery_control #• Adjusts p-values when running multiple statistical hypothesis tests at once (such as testing thousands of genes or features) to control the rate of false positives.

from sklearn.decomposition import SparsePCA #  Performs Sparse Principal Component Analysis to reduce the number of features in a dataset while keeping most of the components at zero, making the results easier to interpret than standard PCA
from sklearn.model_selection import train_test_split #Randomly divides a dataset into separate training and testing subsets so you can evaluate how well a model performs on unseen data.
from sklearn.preprocessing import StandardScaler, LabelEncoder #  Standardizes numerical features by removing the mean and scaling to unit variance, ensuring all features sit on a comparable scale before training a model.
# Converts text or categorical target labels (e.g., "Yes"/"No" or "Cat"/"Dog") into numeric integers (0, 1, etc.) that machine learning algorithms can process.
import pickle # for pickling, reuse StandardScaler/LabelEncoder files • Serializes (saves) Python objects to disk and loads them back later. As noted in your comment, saving a fitted StandardScaler or LabelEncoder lets you apply the exact same scaling and label mappings to new, incoming data in production without refitting.


In [2]:
# Load DataSet
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
## PreProcessing Data
## Removing Irrelevant columns
data = data.drop(['RowNumber','CustomerId','Surname'], axis=1) # axis = 1 for removing column
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
## Encode Categorical Variables
gender_label_encoder = LabelEncoder()
data['Gender'] = gender_label_encoder.fit_transform(data['Gender']) #we used label encoder because gender has only two type male/female
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [5]:
## One Hot encode Geography column we cant use label encoder which will give same value as 0 1

from sklearn.preprocessing import OneHotEncoder #  is used to convert categorical text or integer data into binary numerical columns (1s and 0s) so machine learning models can process them.

ohe_geo = OneHotEncoder()
geo_encoder = ohe_geo.fit_transform(data[['Geography']])

geo_encoder

ohe_geo.get_feature_names_out(['Geography'])

geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns= ohe_geo.get_feature_names_out(['Geography'])) #This line of code converts the raw encoded matrix from OneHotEncoder back into a clean, readable pandas DataFrame with descriptive column names.
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [6]:
## Combine One Hot Encoder Columns with Original Data

data = pd.concat( [data.drop('Geography', axis=1), geo_encoded_df], axis=1 )
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [7]:
## Save the Encoders and Scalers

with open('label_encoder_gender.pkl','wb') as file:   #wb is write byte mode  .pkl is file extension
    pickle.dump(gender_label_encoder, file)

with open('ohe_geo.pkl', 'wb') as file:
    pickle.dump(ohe_geo, file)

In [8]:
## Divide the Data Set into Dependent and Independent feature

X = data.drop('Exited', axis=1)  # independent feature
y = data['Exited']   # Dependent feature

## Split the Data in training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
# X - All the input columns , y - Single column we are trying to predict, 0.2 means 20% of data for testing in X_test and y_test/ 80% for training in X_train, y_train, 42 - Setting a fixed seed number (like 42, a common programming convention) ensures the shuffle happens in the exact same order every time you run your code. Without it, you would get slightly different accuracy scores on every run.


In [9]:
## Scale these features

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential # model
from tensorflow.keras.layers import Dense # for hidden layer node/neuron
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [12]:
(X_train.shape[1],) # Means it is single dimension with 13 elements

(12,)

In [13]:
## Build our ANN Model
model = Sequential( [
    Dense(64, activation = 'relu', input_shape = (X_train.shape[1],) ), #X_train.shape[1] instead of 12 # Hidden layer 1 connected with input layer having 64
                                         # nodes/neurons, activation func type relu and input is shape of 1 means no.of inputs
    Dense(32, activation = 'relu'), # Here we dont need to pass inputshape as we are using sequential it knows it is connected all the layers will be connected
    Dense(1, activation = 'sigmoid')  ## Output layer thats why 1
] )


C:\Users\rohit\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate = 0.01)
loss = tensorflow.keras.losses.BinaryFocalCrossentropy()
loss

<LossFunctionWrapper(<function binary_focal_crossentropy at 0x0000016D48BEE840>, kwargs={'apply_class_balancing': False, 'alpha': 0.25, 'gamma': 2.0, 'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

In [16]:
## In order to do forward and backward propogation we need to compile the model

model.compile(optimizer = opt, loss = "binary_crossentropy", metrics = ['accuracy'] ) # binary crossentropy if we have multi classification then use sparscrossentropy


In [17]:
## Setup tensorboard to capture and visualise the logs while training
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/fit"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S") # make directory for logs
tensorflow_callback = TensorBoard( log_dir=log_dir, histogram_freq =1 )


In [18]:
## Setup Early Stopping to stop the training when the loss value is not decreasing after a while
early_stopping_callback = EarlyStopping(monitor = 'val_loss', patience =5, restore_best_weights = True )  #validation loss - ,
# be patience for 5 epochs after that if find no improvement then stop it there
# To select the best weight while forward/backward propogation


In [19]:
## Train the Model
history = model.fit(
    X_train, y_train, validation_data = (X_test, y_test), epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8335 - loss: 0.3989 - val_accuracy: 0.8550 - val_loss: 0.3509
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8559 - loss: 0.3576 - val_accuracy: 0.8510 - val_loss: 0.3640
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8553 - loss: 0.3492 - val_accuracy: 0.8535 - val_loss: 0.3507
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8580 - loss: 0.3441 - val_accuracy: 0.8540 - val_loss: 0.3431
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8601 - loss: 0.3428 - val_accuracy: 0.8615 - val_loss: 0.3393
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8635 - loss: 0.3380 - val_accuracy: 0.8550 - val_loss: 0.3520
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8629 - loss: 0.3388 - val_accuracy: 0.8605 - val_loss: 0.3473
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8633 - loss: 0.3359 - val_accu

In [20]:
model.save('model.h5')

In [21]:
## Load Tensorboard extension
%load_ext tensorboard

In [22]:
## Take this tensor board along with it use log directory
%tensorboard --logdir logs/fit20261123-165500